In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error
import warnings

warnings.filterwarnings('ignore')

path = '/kaggle/input/competitions/march-machine-learning-mania-2026/'

seeds = pd.concat([pd.read_csv(path + 'MNCAATourneySeeds.csv'), pd.read_csv(path + 'WNCAATourneySeeds.csv')], axis=0)
regular_results = pd.concat([pd.read_csv(path + 'MRegularSeasonCompactResults.csv'), pd.read_csv(path + 'WRegularSeasonCompactResults.csv')], axis=0)
tourney_results = pd.concat([pd.read_csv(path + 'MNCAATourneyCompactResults.csv'), pd.read_csv(path + 'WNCAATourneyCompactResults.csv')], axis=0)
sample_sub = pd.read_csv(path + 'SampleSubmissionStage1.csv')

seeds['Seed_Int'] = seeds['Seed'].apply(lambda x: int(x[1:3]))
seeds_df = seeds[['Season', 'TeamID', 'Seed', 'Seed_Int']]

def get_season_stats(df):
    win_stats = df.groupby(['Season', 'WTeamID']).agg(Wins=('WTeamID', 'count'), W_Pts=('WScore', 'sum'), W_Opp_Pts=('LScore', 'sum')).reset_index().rename(columns={'WTeamID': 'TeamID'})
    loss_stats = df.groupby(['Season', 'LTeamID']).agg(Losses=('LTeamID', 'count'), L_Pts=('LScore', 'sum'), L_Opp_Pts=('WScore', 'sum')).reset_index().rename(columns={'LTeamID': 'TeamID'})
    stats = pd.merge(win_stats, loss_stats, on=['Season', 'TeamID'], how='outer').fillna(0)
    stats['TotalGames'] = stats['Wins'] + stats['Losses']
    stats['WinRate'] = stats['Wins'] / stats['TotalGames']
    stats['Avg_Diff'] = ((stats['W_Pts'] + stats['L_Pts']) - (stats['W_Opp_Pts'] + stats['L_Opp_Pts'])) / stats['TotalGames']
    return stats[['Season', 'TeamID', 'WinRate', 'Avg_Diff']]

season_features = get_season_stats(regular_results)

tourney_results['T1_TeamID'] = np.where(tourney_results['WTeamID'] < tourney_results['LTeamID'], tourney_results['WTeamID'], tourney_results['LTeamID'])
tourney_results['T2_TeamID'] = np.where(tourney_results['WTeamID'] > tourney_results['LTeamID'], tourney_results['WTeamID'], tourney_results['LTeamID'])
tourney_results['Target'] = np.where(tourney_results['WTeamID'] == tourney_results['T1_TeamID'], 1, 0)

train_df = tourney_results[['Season', 'T1_TeamID', 'T2_TeamID', 'Target']].copy()

def merge_features(df):
    for i in [1, 2]:
        df = pd.merge(df, seeds_df, left_on=['Season', f'T{i}_TeamID'], right_on=['Season', 'TeamID'], how='left').drop('TeamID', axis=1)
        df.rename(columns={'Seed_Int': f'T{i}_Seed', 'Seed': f'T{i}_Seed_Full'}, inplace=True)
        df = pd.merge(df, season_features, left_on=['Season', f'T{i}_TeamID'], right_on=['Season', 'TeamID'], how='left').rename(columns={'WinRate': f'T{i}_WinRate', 'Avg_Diff': f'T{i}_ScoreDiff'}).drop('TeamID', axis=1)
    
    df['Seed_Diff'] = df['T1_Seed'] - df['T2_Seed']
    df['WinRate_Diff'] = df['T1_WinRate'] - df['T2_WinRate']
    df['ScoreDiff_Diff'] = df['T1_ScoreDiff'] - df['T2_ScoreDiff']
    df['Is_PlayIn'] = df['T1_Seed_Full'].str[:3] == df['T2_Seed_Full'].str[:3]
    return df

train_df = merge_features(train_df).fillna(0)

features = ['T1_Seed', 'T2_Seed', 'Seed_Diff', 'T1_WinRate', 'T2_WinRate', 'WinRate_Diff', 'T1_ScoreDiff', 'T2_ScoreDiff', 'ScoreDiff_Diff']
val_seasons = [2022, 2023, 2024, 2025] 
models = []
all_season_mse = []

xgb_params = {'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'learning_rate': 0.03, 'max_depth': 4, 'random_state': 42}

for season in val_seasons:
    X_train = train_df[train_df['Season'] < season][features]
    y_train = train_df[train_df['Season'] < season]['Target']
    
    val_mask = (train_df['Season'] == season) & (~train_df['Is_PlayIn'])
    X_val = train_df[val_mask][features]
    y_val = train_df[val_mask]['Target']
    
    if len(X_val) == 0: continue
    
    model = xgb.train(xgb_params, xgb.DMatrix(X_train, label=y_train), num_boost_round=1000, 
                      evals=[(xgb.DMatrix(X_val, label=y_val), 'val')], early_stopping_rounds=50, verbose_eval=False)
    
    clipped_preds = np.clip(model.predict(xgb.DMatrix(X_val)), 0.025, 0.975)
    season_mse = mean_squared_error(y_val, clipped_preds)
    
    all_season_mse.append(season_mse)
    models.append(model)
    
    print(f"Season {season} True LB MSE: {season_mse:.5f} (计分场次: {len(y_val)})")

print(f"\n==== 终极对齐平均 CV MSE: {np.mean(all_season_mse):.5f} ====")

sub_df = sample_sub.copy()
sub_df['Season'] = sub_df['ID'].apply(lambda x: int(x.split('_')[0]))
sub_df['T1_TeamID'] = sub_df['ID'].apply(lambda x: int(x.split('_')[1]))
sub_df['T2_TeamID'] = sub_df['ID'].apply(lambda x: int(x.split('_')[2]))

test_df = merge_features(sub_df).fillna(0)
test_preds = np.zeros(len(test_df))

for idx, season in enumerate(val_seasons):
    season_mask = test_df['Season'] == season
    dtest_season = xgb.DMatrix(test_df[season_mask][features])
    pred_season = models[idx].predict(dtest_season)
    test_preds[season_mask] = pred_season

sub_df['Pred'] = np.clip(test_preds, 0.025, 0.975)
sub_df[['ID', 'Pred']].to_csv('submission.csv', index=False)

Season 2022 True LB MSE: 0.19495 (计分场次: 126)
Season 2023 True LB MSE: 0.18286 (计分场次: 126)
Season 2024 True LB MSE: 0.15434 (计分场次: 126)
Season 2025 True LB MSE: 0.12538 (计分场次: 126)

==== 终极对齐平均 CV MSE: 0.16438 ====
